# Run and submit your planner
Edit [my_submission/student_planner.py](../my_submission/student_planner.py): leave exactly one search, reactive or custom factory active. Start a fresh kernel. This notebook runs your saved files and exports their results.


In [ ]:
from pathlib import Path
from planner.common.scenarios import ASSETS
from planner.runtime.artifacts import export_result
from planner.runtime.evaluation import describe_result
from planner.runtime.execution import run_submission

root = Path.cwd()
if root.name == "tutorials":
    root = root.parent
submission = root / "my_submission"
scenarios = [
    ASSETS / "scenarios/public/DEU_AugsburgCRC26P-23_16300_I-1.xml",
    ASSETS / "scenarios/public/DEU_FlensburgCRC26E-71_11400_I-1.xml",
    ASSETS / "scenarios/public/DEU_ErlangenCRC26P-12_17300_I-1.xml",
]
submission


## Reuse a search class
Replace `AStarSearch` in the import and `smp_spec(AStarSearch)` call with your class. Keep its constructor and `execute_search()` interface:

```python
from SMP.motion_planner.search_algorithms.best_first_search import AStarSearch
from planner.spec import smp_spec

def create_planner():
    return smp_spec(AStarSearch)
```

Keep submitted classes and helpers in saved Python modules. An optional complete `SMP/` tree beside `student_planner.py` overrides the bundled search code, including absolute `SMP.*` imports.

The adapter preserves the actual initial state. Off-grid speeds need a feasible five-step connection to the primitive library; unsupported steering or speed transitions fail explicitly. [Notebook 00](00_interactive_search_quickstart.ipynb#Inspect-the-drive) explains the adapter's state, horizon and execution behavior.


## Run the starters
Augsburg, Flensburg and Erlangen are included in the 100 public tasks. `run_submission()` fixes SM1, seed 1234 and the 200-step allowance. Planner configuration belongs in your factory.


In [ ]:
results = {}
for scenario in scenarios:
    result = run_submission(scenario, submission)
    export_result(result, root / "solutions")
    results[scenario.stem] = describe_result(result)
results


## Use a custom planner
Activate the custom option and implement `StudentPlanner.plan()` in [custom_planner.py](../my_submission/custom_planner.py), or use your own modules. Only `plan(scenario, planning_problem) -> Trajectory` is required; no inheritance is needed. For a class in an adjacent `my_planner.py`:

```python
from planner.spec import PlannerSpec
from my_planner import StudentPlanner

def create_planner() -> PlannerSpec:
    return PlannerSpec(StudentPlanner())
```

The factory runs once per scenario; `plan()` receives updated observations repeatedly. Use `planning_problem.initial_state` for the current ego state and `planning_problem.goal` for the destination. You can retain state between calls and optionally implement `reset()` for reuse with `run_spec()`.

| Contract | Requirement |
| --- | --- |
| Output | Vehicle-center positions, consecutive absolute timestamps from the current or next step, and at least one future state |
| Observation | Copied inputs with recorded future traffic withheld; supply your own prediction/planning logic |
| Vehicle | `PlannerSpec` defaults to KS/FORD_ESCORT, 4.298 × 1.674 m, one-step replanning. Use `SimulationSettings.for_vehicle(...)` and the same `VehicleType` in the spec |
| Final Solution | Framework-generated from the driven trajectory, original scenario/problem IDs and spec vehicle metadata; all validity checks apply |


Return enough future states to cover the configured execution interval. The supplied **route-following dummy** samples the reference path at constant speed and stops at its endpoint. It does not react to traffic or guarantee dynamic feasibility, road-boundary compliance, or goal arrival. [Tutorial 02](02_example_planners.ipynb#4.-Follow-the-route-with-a-constant-speed-dummy) shows its behavior; add motion selection and validation before using it as a competition solution.

The optional helpers in `custom_planner.py` are suggestions:

| Helper | Provided example |
| --- | --- |
| `plan_route()` | CommonRoad Route Planner, returning an `(N, 2)` reference path |
| `predict_obstacles()` | Lane-following prediction with straight-line fallback |
| `is_trajectory_valid()` | A `pass` stub for dynamics, boundary, and collision checks |

The dummy calls `plan_route()`; it does not call the prediction or validation helpers. Reuse, replace or move them into your own modules and classes. `custom_planner.py` is only needed for the custom option and can be renamed with its import. The next section explains prediction and its time convention.


## Use and improve the predictor
Both supplied search and reactive adapters use `predict_observations()`. This shared baseline follows lanes at current speed with straight-line fallback; it does not anticipate braking or lane changes. Inside your custom `plan()`:

```python
from planner.common.prediction import predict_observations

current = planning_problem.initial_state.time_step
predicted = predict_observations(scenario, current, horizon=20)
```

Build collision checks against `predicted`. Time `0` means now; future states are `1..20`. Align candidate ego and goal times to that origin, then return absolute ego timestamps.

A replacement predictor must preserve the map, static obstacles, observed actor IDs/shapes and timestep. Use current observations, retained observation history and map geometry; recorded futures and SUMO routes are unavailable. The supplied adapters call the default helper directly: adding a predictor module alone does not activate it. Include a planner that calls your predictor.

Compare validity, SM1 and runtime on the same scenarios. Final validation uses actual simulated traffic.


## Inspect failures
`evaluate_scenario()` reports failures without stopping a batch: `no_path`, `incompatible_state`, `invalid_output` and `planner_exception` distinguish planning causes. `scoring_error` means the drive completed and passed validation but could not receive an SM1 score. This temporary example returns no trajectory.


In [ ]:
from dataclasses import asdict
from tempfile import TemporaryDirectory
from planner.runtime.evaluation import evaluate_scenario

with TemporaryDirectory(prefix="commonroad-broken-example-") as temporary:
    broken = Path(temporary)
    (broken / "student_planner.py").write_text("""from planner.spec import PlannerSpec

class StudentPlanner:
    def plan(self, scenario, problem):
        return None

def create_planner():
    return PlannerSpec(StudentPlanner())
""")
    failure = asdict(evaluate_scenario(scenarios[0], broken))
failure


## Package and submit
Keep required configs, model weights and data inside your submission folder. In your planner module, read them relative to `__file__`, for example for an adjacent `config.json`:

```python
import json
from pathlib import Path

config = json.loads((Path(__file__).parent / "config.json").read_text())
```

The loader, ZIP and source fingerprint use the same files. After running your saved planner, package its source and assets separately from the exported result folders:


In [ ]:
from planner.runtime.submission import package_submission

package_submission(submission, root / "submission.zip")


A failed run may leave an older result folder; check the run output before submitting. [Notebook 03](03_solution_and_cost_validation.ipynb) shows how to validate saved pairs.

Use the [main README](../README.md#save-results-for-submission) for the submission contents and environment requirements, and [tutorial 05](05_batch_processing.ipynb) to run the full public set sequentially or in parallel with progress and result reports.
